
# [🧅 Introduction to Subqueries](https://learn.microsoft.com/en-us/training/modules/write-subqueries/1-introduction)

---

## Simplifying Complex Queries
Sometimes, when using Transact-SQL to retrieve data from a database, a single query can become overly complex and difficult to read or maintain. In these cases, it is often easier to simplify the logic by breaking the query down into multiple, simpler queries that work together to achieve the desired result. This is achieved using **Subqueries**, where an *inner query* computes a result and passes it directly to an *outer query*.


## What is a Subquery?
Transact-SQL supports the creation of **subqueries**. A subquery is simply a query nested inside another query. The outer query uses the results of the inner query to evaluate its own `WHERE`, `HAVING`, or `SELECT` clauses.

- The **inner query** (the subquery) executes first.
- It returns its result (a value, a list of values, or a table) to the **outer query**.
- The outer query then uses that result to complete its own execution and return the final dataset.

## The Execution Flow of a Subquery

When SQL Server processes a query containing a subquery, it follows a specific logical order. The inner query must be fully evaluated before the outer query can use its results.

```mermaid
graph TD
    classDef inner fill:#e3f2fd,stroke:#1565c0,stroke-width:2px;
    classDef pass fill:#f3e5f5,stroke:#7b1fa2,stroke-width:2px;
    classDef outer fill:#fff3e0,stroke:#ef6c00,stroke-width:2px;
    classDef result fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px;

    A["1. Inner Query (Subquery)\nExecutes first"]:::inner
    B["Returns intermediate result\n(Scalar value, List, or Table)"]:::pass
    C["2. Outer Query\nReceives the result and applies it\n(e.g., in a WHERE or SELECT clause)"]:::outer
    D["3. Final Result Set\nReturned to the user"]:::result

    A --> B --> C --> D
```
> **Analogy:** Think of a subquery like a mathematical expression inside parentheses: $(2 + 3) * 4$. You must solve the inner part $(2 + 3 = 5)$ before you can multiply it by the outer part $5 * 4 = 20$.



***

## Module Learning Objectives

In this module, you will master the techniques for writing and utilizing subqueries in T-SQL. By the end of this module, you will be able to:

1. **Understand Subqueries:** Grasp the syntax and logic of nesting a `SELECT` statement inside another `SELECT`, `INSERT`, `UPDATE`, or `DELETE` statement.
2. **Scalar vs. Multi-Valued Subqueries:** 
    * **Scalar:** Returns exactly one single value (one row, one column).
    * **Multi-valued:** Returns a single column with multiple rows (often used with the `IN` operator).
3. **Self-Contained vs. Correlated Subqueries:**
    * **Self-contained:** The inner query can run independently of the outer query. It executes once.
    * **Correlated:** The inner query relies on data from the outer query. It executes repeatedly (once for every row evaluated by the outer query).

# [🧅 Understanding Subqueries](https://learn.microsoft.com/en-us/training/modules/write-subqueries/2-understand-subqueries)


## What is a Subquery?
A **subquery** is simply a `SELECT` statement nested (or embedded) within another query. Being able to nest queries within other queries greatly enhances your ability to write effective, modular, and readable T-SQL code.

### The Anatomy of a Subquery
When you write a subquery, you are working with two distinct parts:

1. **The Inner Query (The Subquery)**: The nested query that executes first. Its purpose is to retrieve intermediate data.

2. **The Outer Query**: The main query that contains the inner query. It receives the results from the inner query and uses them to produce the final result set.

> **General Rule:** In most cases, a subquery is evaluated **once**, and its results are passed up to the outer query to complete the execution.

***

## Types of Subqueries by Return Value

The form of the results returned by the inner query determines how the outer query must process it. Subqueries are classified into two types based on their output:

### 1. Scalar Subqueries
* **Output:** Returns a **single value** (one row and one column).
* **Usage:** Often used with standard comparison operators ( `=`, `>`, `<` ) in the `WHERE` or `SELECT` clauses.
* *Example logic:* "Find all products that cost more than the *[average price of all products]*."

### 2. Multi-Valued Subqueries

* **Output:** Returns a **single column with multiple rows** (essentially a list).
* **Usage:** Because it returns multiple values, you cannot use an `=` sign. The outer query must use list-compatible operators like `IN` or `NOT IN`.
* *Example logic:* "Find all customers whose IDs are in the list of *[customers who bought a bike in 2022]*."



### Visualizing the Output

```mermaid
graph TD
    classDef inner fill:#e3f2fd,stroke:#1565c0,stroke-width:2px;
    classDef scalar fill:#c8e6c9,stroke:#2e7d32,stroke-width:2px;
    classDef multi fill:#fff3e0,stroke:#ef6c00,stroke-width:2px;
    classDef outer fill:#f3e5f5,stroke:#7b1fa2,stroke-width:2px;

    A["Inner Query\n(Subquery)"]:::inner --> B{"What is the output shape?"}
    
    B -->|Single Row, Single Column| C["Scalar Subquery\nReturns: 1 Value"]:::scalar
    B -->|Multiple Rows, Single Column| D["Multi-Valued Subquery\nReturns: List of Values"]:::multi
    
    C --> E["Outer Query uses =, >, <, etc."]:::outer
    D --> F["Outer Query uses IN, EXISTS, etc."]:::outer
```

***

## Types of Subqueries by Dependency

In addition to their output shape, subqueries are also classified by whether they rely on the outer query to function. 

### 1. Self-Contained Subqueries
- **Definition**: These subqueries have **no dependencies** on the outer query. 
- **Execution**: They can be written, highlighted, and executed as completely stand-alone queries. 
- **Performance**: Because they are independent, the SQL Server engine evaluates them **only once**, passes the result to the outer query, and moves on.

### 2. Correlated Subqueries
- **Definition**: These subqueries reference one or more columns from the **outer query**. They are inherently dependent on the outer query.
- **Execution**: They **cannot** be run separately. If you try to execute just the inner query, it will throw an error because the outer columns don't exist in its scope.
- **Performance**: They are evaluated **repeatedly**—once for every single row processed by the outer query. 

```mermaid
graph TD
    subgraph Query Structure
        A[Outer Query] -->|Executes and evaluates using| B((Inner Query <br> Subquery))
        B -->|Passes Results| A
    end

    subgraph Classifications
        C[Subquery Types] --> D[By Result Format]
        C --> E[By Dependency]

        D --> D1[Scalar <br> Returns exactly ONE single value]
        D --> D2[Multi-Valued <br> Returns a single-column list of values]

        E --> E1[Self-Contained <br> Independent, runs only once]
        E --> E2[Correlated <br> References outer query, runs repeatedly]
    end
    
    style A fill:#e1f5fe,stroke:#0288d1,stroke-width:2px
    style B fill:#fff3e0,stroke:#f57c00,stroke-width:2px
```

### Execution Flow Comparison
```mermaid
flowchart TD
    classDef self fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px;
    classDef corr fill:#ffebee,stroke:#c62828,stroke-width:2px;
    classDef outer fill:#f3e5f5,stroke:#7b1fa2,stroke-width:2px;

    subgraph Self-Contained Flow
        direction TB
        S1["1. Run Inner Query\n(Independent)"]:::self --> S2["2. Get Result"]:::self
        S2 --> S3["3. Pass to Outer Query\n(Evaluated ONCE)"]:::self
    end
    
    subgraph Correlated Flow
        direction TB
        C1["1. Outer Query grabs Row 1"]:::outer --> C2["2. Inner Query runs\nusing Row 1's data"]:::corr
        C2 --> C3["3. Inner Query passes result\nback to Outer Query"]:::corr
        C3 --> C4["4. Outer Query moves to Row 2\n(Repeat for EVERY row)"]:::outer
    end
```

***

## Summary & Quick Reference

Subqueries are a powerful tool for breaking down complex logic. Here is a quick reference matrix for the types of subqueries you will encounter:

| Classification | Type | Description | Execution |
| :--- | :--- | :--- | :--- |
| **By Output** | **Scalar** | Returns exactly 1 value. | Outer query uses `=`, `>`, `<`, etc. |
| **By Output** | **Multi-Valued** | Returns a list of values (1 column). | Outer query uses `IN`, `ANY`, `ALL`. |
| **By Dependency** | **Self-Contained** | Independent of the outer query. | Evaluated **once**. Can be run alone. |
| **By Dependency** | **Correlated** | Depends on columns from the outer query. | Evaluated **row-by-row**. Cannot be run alone. |



# [🎯 Using Scalar and Multi-Valued Subqueries](https://learn.microsoft.com/en-us/training/modules/write-subqueries/3-scalar-multi-values-subqueries)

When writing subqueries, the database engine must know what "shape" of data to expect back from the inner query. The two primary shapes are **Scalar** (a single value) and **Multi-Valued** (a list of values).

```mermaid
graph TD
    A[Inner Query Execution] --> B{What shape is the result?}
    
    B -->|1 Row, 1 Column| C[Scalar Subquery]
    B -->|Multiple Rows, 1 Column| D[Multi-Valued Subquery]
    
    C --> E[Use with: =, >, <, <br> or in SELECT/WHERE clauses]
    D --> F[Use with: IN, NOT IN]
    
    style C fill:#e1f5fe,stroke:#0288d1,stroke-width:2px
    style D fill:#fff3e0,stroke:#f57c00,stroke-width:2px
```

## 1. Scalar Subqueries
A **scalar subquery** returns exactly one single value, i.e., **one row and one column**. 

Because it returns a single value, a scalar subquery can be used anywhere in an outer T-SQL statement where a single-valued expression is permitted:
- `WHERE` clause
- `HAVING` clause
- `SELECT` list (as a calculated column)
- `FROM` clause
- Data modification statements (`UPDATE`, `DELETE`, `INSERT`)

### Example Scenario
Suppose you want to retrieve the details of the very last order placed. You can assume the last order has the highest `SalesOrderID`. 
1. First, find the highest ID.
2. Then, use that ID to filter the order details.

***

**Scalar Subquery Examples**
```sql
-- ==========================================
-- SCALAR SUBQUERY IN A WHERE CLAUSE
-- ==========================================
-- The inner query returns a single value (the MAX SalesOrderID).
-- The outer query uses that single value to filter the results.

SELECT SalesOrderID, ProductID, OrderQty
FROM Sales.SalesOrderDetail
WHERE SalesOrderID = 
   (SELECT MAX(SalesOrderID)
    FROM Sales.SalesOrderHeader);


-- ==========================================
-- SCALAR SUBQUERY IN A SELECT LIST
-- ==========================================
-- We can compare the order quantity of the last order 
-- against the average order quantity of ALL orders.
-- The inner query runs once and acts as a constant column.

SELECT SalesOrderID, ProductID, OrderQty,
    (SELECT AVG(OrderQty)
     FROM SalesLT.SalesOrderDetail) AS AvgQty
FROM SalesLT.SalesOrderDetail
WHERE SalesOrderID = 
    (SELECT MAX(SalesOrderID)
     FROM SalesLT.SalesOrderHeader);
```

***

## Guidelines for Writing Scalar Subqueries

When writing scalar subqueries, keep these crucial rules in mind:

1. **Parentheses are Mandatory**: Always enclose the subquery in parentheses `()` to denote it as a distinct nested query.
2. **Nesting Limits**: T-SQL supports up to **32 levels** of nested subqueries (though we will focus on 2-level queries in this module).
3. **Handling Empty Sets (NULLs)**: If the inner query returns no rows (an empty set), the result of the subquery is `NULL`. Ensure your outer query can gracefully handle a `NULL` value if this scenario is possible.
4. **Single Column Rule**: The inner query should generally return a **single column**. Selecting multiple columns in a scalar subquery will result in an error. *(The only exception to this rule is when using the `EXISTS` keyword).*

***

## 2. Multi-Valued Subqueries

A **multi-valued subquery** returns **multiple rows**, but still only **one column**. It acts much like a single-column temporary table.

Because it returns a list of values, multi-valued subqueries are most commonly used with the **`IN`** operator in a `WHERE` clause.

### Example Scenario
You want to find all sales orders placed by customers who live in Canada. 
1. The inner query finds the list of all `CustomerID`s located in Canada.
2. The outer query filters the `SalesOrderHeader` table to only include orders matching those `CustomerID`s.

***

**Multi-Valued Subquery Example**
```sql
-- ==========================================
-- MULTI-VALUED SUBQUERY WITH IN()
-- ==========================================
-- The inner query returns a list of CustomerIDs in Canada.
-- The outer query returns orders for those specific customers.

SELECT CustomerID, SalesOrderID
FROM Sales.SalesOrderHeader
WHERE CustomerID IN (
    SELECT CustomerID
    FROM Sales.Customer
    WHERE CountryRegion = 'Canada'
);
```

***

## Subqueries vs. JOINs: When to use which?

Many multi-valued subqueries can easily be rewritten using a `JOIN`. In fact, the SQL Server Query Optimizer will often internally convert a simple subquery into a `JOIN` anyway, meaning there is no performance difference between the two.

### How to Decide?
The choice often comes down to readability, personal preference, and specific output requirements.

```mermaid
flowchart TD
    classDef join fill:#c8e6c9,stroke:#2e7d32,stroke-width:2px;
    classDef sub fill:#bbdefb,stroke:#1565c0,stroke-width:2px;
    classDef decision fill:#fff3e0,stroke:#ef6c00,stroke-width:2px;

    A["Need to retrieve data from multiple tables"] --> B{"Do you need columns from BOTH tables in the final SELECT list?"}:::decision
    
    B -->|Yes| C["Use a JOIN"]:::join
    B -->|No| D{"Is the logic highly complex or step-by-step?"}:::decision
    
    D -->|Yes| E["Use a Subquery\n(Breaks logic into readable steps)"]:::sub
    D -->|No| F["Either works fine!\nOptimizer will likely treat them the same."]:::join
```

### ⚠️ Crucial Restriction of Subqueries
When using a nested subquery, **the final result set returned to the client can ONLY include columns from the outer query.** 
- If you need to display columns from *both* tables side-by-side (e.g., Customer Name AND Order Date), you **must** use a `JOIN`.
- If you only need to filter the outer table based on the inner table (e.g., Orders WHERE Customer is in Canada), a subquery is perfect.

***

**Subquery vs JOIN Comparison**
```sql
-- ==========================================
-- APPROACH 1: MULTI-VALUED SUBQUERY
-- ==========================================
-- Great for filtering. We only return columns from the outer table.
SELECT CustomerID, SalesOrderID
FROM Sales.SalesOrderHeader
WHERE CustomerID IN (
    SELECT CustomerID
    FROM Sales.Customer
    WHERE CountryRegion = 'Canada'
);


-- ==========================================
-- APPROACH 2: EQUIVALENT JOIN
-- ==========================================
-- Required if we want to see columns from BOTH tables in the output.
SELECT c.CustomerID, o.SalesOrderID, c.CountryRegion
FROM Sales.Customer AS c
JOIN Sales.SalesOrderHeader AS o
    ON c.CustomerID = o.CustomerID
WHERE c.CountryRegion = 'Canada';
```

***

## Summary

- **Scalar Subqueries** return a single value (1 row, 1 column). They can be used in `SELECT`, `WHERE`, `HAVING`, and data modification statements.
- **Multi-Valued Subqueries** return a list of values (multiple rows, 1 column). They are typically used with the `IN()` operator.
- **Subquery Rules**: 
  - Must be enclosed in parentheses `()`.
  - Must return a single column (unless using `EXISTS`).
  - Returns `NULL` if the inner query yields an empty set.
- **Subqueries vs. JOINs**: 
  - Use a **JOIN** when you need to return columns from multiple tables in your final output.
  - Use a **Subquery** when you want to filter an outer query based on complex, step-by-step logic derived from another table, and you only need columns from the outer table in your final output.



# [🔄 Self-Contained vs. Correlated Subqueries](https://learn.microsoft.com/en-us/training/modules/write-subqueries/4-self-contained-correlated-subqueries)


## Recap: Self-Contained Subqueries
Previously, we looked at **self-contained subqueries**. In these queries, the inner query is completely independent of the outer query. It executes **only once**, and its results are passed to the outer query.

## Correlated Subqueries
T-SQL also supports **correlated subqueries**. A correlated subquery references one or more columns from the outer query. Because of this dependency, **it cannot be executed separately** from the outer query, which makes testing and debugging slightly more complex.

### How it Works Logically
Because the inner query relies on a value from the outer query, it cannot be executed on its own. Conceptually, the outer query runs first, and **for each row returned by the outer query, the inner query is processed**.

### Execution Flow Comparison
```mermaid
flowchart TD
    classDef self fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px;
    classDef corr fill:#ffebee,stroke:#c62828,stroke-width:2px;
    classDef outer fill:#f3e5f5,stroke:#7b1fa2,stroke-width:2px;

    subgraph Self-Contained Execution
        direction TB
        S1["1. Inner Query Runs\n(Independent)"]:::self --> S2["2. Returns Result"]:::self
        S2 --> S3["3. Outer Query Runs\n(Uses result once)"]:::self
    end
    
    subgraph Correlated Execution
        direction TB
        C1["1. Outer Query grabs Row 1"]:::outer --> C2["2. Inner Query Runs\n(Uses Row 1's data)"]:::corr
        C2 --> C3["3. Outer Query evaluates Row 1"]:::outer
        C3 --> C4["4. Outer Query grabs Row 2"]:::outer
        C4 --> C5["5. Inner Query Runs Again\n(Uses Row 2's data)"]:::corr
        C5 --> C6["... Repeats for EVERY row ..."]:::corr
    end
```


***

**Correlated Subquery Example**
```sql
-- ==========================================
-- CORRELATED SUBQUERY
-- ==========================================
-- Goal: Return the most recent order (highest SalesOrderID) for EACH customer.
-- The inner query references the CustomerID from the outer query (o1).

SELECT SalesOrderID, CustomerID, OrderDate
FROM SalesLT.SalesOrderHeader AS o1 -- Outer query alias
WHERE SalesOrderID =
    (SELECT MAX(SalesOrderID)
     FROM SalesLT.SalesOrderHeader AS o2 -- Inner query alias
     WHERE o2.CustomerID = o1.CustomerID) -- The Correlation!
ORDER BY CustomerID, OrderDate;
```

***

## Guidelines for Writing Correlated Subqueries

When writing correlated subqueries, follow these steps to ensure clarity and correctness:

1. **Plan for the Return Result**: Determine if the inner query will return a single value (scalar) or multiple values. 
   - If scalar, use operators like `=`, `<`, `>`, `<>`.
   - If multi-valued, use the `IN` predicate.
   - Always plan to handle potential `NULL` results.
2. **Alias the Outer Table**: Declare an alias for the table in the outer query so you can reference its columns inside the inner query.
3. **Alias the Inner Table**: Create an alias for the inner table to distinguish it from the outer table.
4. **Establish the Correlation**: Write the inner query's `WHERE` clause to compare its columns against the outer query's aliased columns. This linkage is what makes it "correlated."

***

# Working with `EXISTS` and `NOT EXISTS`

Sometimes, you don't actually need the *data* returned by a subquery; you only need to know **if any matching rows exist**. 

The **`EXISTS`** predicate checks whether any rows meet a specified condition. Instead of returning data, it simply returns **`TRUE`** or **`FALSE`**.

### Why use `EXISTS`?
It is highly efficient for validating data. SQL Server can "short-circuit" the evaluation. As soon as it finds the **first** matching row, it returns `TRUE` and stops scanning. It does not need to count every single occurrence.

### Visualizing the Performance Difference
```mermaid
sequenceDiagram
    participant Outer as Outer Query
    participant Sub as Subquery
    
    Note over Outer,Sub: Scenario: Checking if Customer has ANY orders
    
    Outer->>Sub: Does Customer 1 have orders?
    Sub-->>Sub: Scans Order Table...
    Sub-->>Sub: Finds Order #101!
    Sub-->>Outer: Returns TRUE (Stops scanning immediately)
    
    Note over Outer,Sub: If we used COUNT(*) > 0 instead...
    Outer->>Sub: Count ALL orders for Customer 1
    Sub-->>Sub: Scans Order Table...
    Sub-->>Sub: Finds Order #101, #102, #103, #104...
    Sub-->>Sub: Finishes scanning entire table
    Sub-->>Outer: Returns Count = 4
    Outer->>Outer: Evaluates 4 > 0 (TRUE)
```


***

**⚡ Performance Benefit: Short-Circuiting (EXISTS vs. COUNT(*))**
1. Consider these two approaches to finding customers who have placed at least one order:
    - **The `COUNT` Approach (Slower)**: Why it's slower: SQL Server must find every single order for a customer, tally them all up, and then check if the total is greater than 0.
    - **The `EXISTS` Approach (Faster)**: Why it's faster: As soon as SQL Server finds one matching order, it instantly returns TRUE and stops searching for that customer. It doesn't waste time counting the rest.
```sql
-- ==========================================
-- APPROACH 1: Using COUNT(*) (Less Efficient)
-- ==========================================
-- The subquery must count EVERY order for the customer, 
-- even if they have thousands of orders.
SELECT CustomerID, CompanyName, EmailAddress 
FROM SalesLT.Customer AS c 
WHERE
    (SELECT COUNT(*) 
     FROM SalesLT.SalesOrderHeader AS o
     WHERE o.CustomerID = c.CustomerID) > 0;


-- ==========================================
-- APPROACH 2: Using EXISTS (More Efficient)
-- ==========================================
-- The subquery stops scanning the moment it finds the FIRST order.
SELECT CustomerID, CompanyName, EmailAddress 
FROM SalesLT.Customer AS c 
WHERE EXISTS
    (SELECT * 
     FROM SalesLT.SalesOrderHeader AS o
     WHERE o.CustomerID = c.CustomerID);
```

***

## Negating with `NOT EXISTS`

Another incredibly useful application of `EXISTS` is negating it with **`NOT`**. This is the standard, highly-optimized way to find rows in one table that have **no match** in another table (often called an "Anti-Semi-Join").

If a match is found, `NOT EXISTS` evaluates to `FALSE` and immediately stops scanning. If the scan completes without finding a match, it evaluates to `TRUE`.

***

**NOT EXISTS Example**
```sql
-- ==========================================
-- FINDING NON-MATCHES WITH NOT EXISTS
-- ==========================================
-- Returns customers who have NEVER placed an order.
-- Highly efficient: stops scanning as soon as it finds an order for that customer.

SELECT CustomerID, CompanyName, EmailAddress 
FROM SalesLT.Customer AS c 
WHERE NOT EXISTS
    (SELECT * 
     FROM SalesLT.SalesOrderHeader AS o
     WHERE o.CustomerID = c.CustomerID);
```

***

## Guidelines for Writing `EXISTS` Subqueries

When using `EXISTS` or `NOT EXISTS`, follow these specific rules:

1. **Placement**: The keyword `EXISTS` directly follows the `WHERE` clause. No column name or expression precedes it (unless you are using `NOT EXISTS`).
   ```sql
   -- Correct
   WHERE EXISTS (...)
   WHERE NOT EXISTS (...)
   
   -- Incorrect
   WHERE CustomerID = EXISTS (...)
   ```

2. **Use `SELECT *`**: Within the subquery, always use `SELECT *`. 
   - Because `EXISTS` only checks for the *presence* of rows, the actual columns returned are completely irrelevant. 
   - Using `SELECT *` clearly communicates your intent to the reader and the query optimizer.

> ⚠️ **Crucial Warning**: If you are converting a `COUNT(*) > 0` query to an `EXISTS` query, make sure you change the inner query to `SELECT *`. If you leave it as `SELECT COUNT(*)`, the subquery will *always* return exactly one row (the count), meaning `EXISTS` will **always** evaluate to `TRUE`, breaking your logic!

***

## Summary

- **Correlated Subqueries** reference columns from the outer query and conceptually execute once for every row processed by the outer query.
- They are useful for row-by-row comparisons, such as finding the "maximum" or "most recent" record within a specific group.
- **`EXISTS`** is a powerful predicate that checks for the presence of rows, returning `TRUE` or `FALSE`. It is highly optimized because it stops scanning as soon as a match is found.
- **`NOT EXISTS`** is the preferred method for finding records in one table that have no corresponding records in another table.
- When using `EXISTS`, always use `SELECT *` in the inner query, as the returned columns do not matter.

